In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
data_path = "../data/training_taxi_data.parquet"
df = pd.read_parquet(data_path)

print("="*50)
print(f"Shape: {df.shape}")
print("="*50)
df.head()

---
** Feature Selection & Preprocessing **

Based on the EDA and "domain knowledge" (not really haha :p), select the three possibly impactful features for predicting *tip_amount*:

- fare_amount: High correlation
- trip_distance: effort/duration;
- PULocationID: Socio-economic context.

In [ ]:
# Select the specific features and target
selected_features = ['fare_amount', 'trip_distance', 'PULocationID']
target = 'tip_amount'

# create a copy to avoid SettingWithCopy warnings
X = df[selected_features].copy()
y = df[target].copy()

# Densure data integrity
# NOTE: (In a production pipeline, imputation might be preferred, but dropping is safer for training analysis)
initial_len = len(X)
X = X.dropna()
y = y.loc[X.index]
print(f"Preprocessing: Dropped {initial_len - len(X)} rows with missing values.")


print(f"Final Features Shape: {X.shape}")
display(X.head())

---
** Train-Validation-Test Split: 80/10/10 **

In [ ]:
from sklearn.model_selection import train_test_split

# Train vs (val + test) |80/20
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# val vs test | 10/10
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Training Set:   {X_train.shape[0]} samples")
print(f"Validation Set: {X_val.shape[0]} samples")
print(f"Test Set:       {X_test.shape[0]} samples")

---
** Model Selection & Configuration (XGBoost) **

First choice: XGBoost

Justification: It is a tree-based ensemble method that handles non-linear relationships (like fare vs tip) significantly better than linear regression and is more robust to outliers (dataset full of outliers). 

Configuration: We configure it with early_stopping_rounds to automatically stop training when the validation error stops improving, effectively "tuning" the number of trees dynamically.

===========================================================================

*!!!NOTE!!! Usually, for good practice, I would use a basic linear regression model as a baseline, and multiple other stronger models for comparison, but that doesn't seem to be the practice's focus.*

In [ ]:
from xgboost import XGBRegressor

# Initialize the XGBoost Regressor
model = XGBRegressor(
    n_estimators=1000, # sets the maximum potential trees, but we will stop early if needed.
    max_depth=6, # robust default for tabular data to capture complexity without massive overfitting.
    learning_rate=0.05,
    objective='reg:squarederror',
    n_jobs=-1,
    early_stopping_rounds=50,  # not fine-tuned for practicity
    eval_metric="rmse",
    random_state=42
)

print("Model initialized: XGBRegressor")

---
** Evaluation Metrics Utility **

Utility tool to calculate standard regression metrics:

- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)
- R² (Coefficient of Determination)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(model, X_data, y_true, dataset_name="Dataset"):
    """
    Calculates and prints RMSE, MAE, and R2 for a given dataset.
    """
    y_pred = model.predict(X_data)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- {dataset_name} Performance ---")
    print(f"RMSE: ${rmse:.4f}")
    print(f"MAE:  ${mae:.4f}")
    print(f"R²:    {r2:.4f}")
    print("-" * 30)
    return rmse, mae, r2

---
** Training, Tuning (Early Stopping), and Final Testing **

- Train the model. 
- val_set validation after every tree is added. Loss patience=50.
- Report score on the Test set.

In [ ]:
print("Starting training with Early Stopping...")

# Train the model
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],    
    verbose=100  # Print progress every 100 trees
)

print("\n" + "="*50)
print("FINAL EVALUATION")
print("="*50)

# Evaluate on all sets to see the progression
evaluate_model(model, X_train, y_train, "Training")
evaluate_model(model, X_val, y_val, "Validation")
evaluate_model(model, X_test, y_test, "TEST SET")

# Feature Importance Plot
plt.figure(figsize=(10, 5))
importances = pd.Series(model.feature_importances_, index=selected_features).sort_values(ascending=True)
importances.plot(kind='barh', color='teal')
plt.title("Feature Importance (Tip Amount)")
plt.xlabel("Relative Importance")
plt.show()